In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

try:
    from sklearn.manifold import TSNE
    HAS_TSNE = True
except Exception:
    HAS_TSNE = False

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

try:
    from scipy.cluster.hierarchy import linkage, dendrogram
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/task_geometry_atlas"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
NPZ_DIR = os.path.join(OUT_DIR, "npz")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Sample sizes: enough for useful geometry, not too heavy.
N_GSM8K = 30
N_STRATEGYQA = 30
N_MMLU_PER_SUBJECT = 12
MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
    "formal_logic",
    "high_school_mathematics",
]

# Which layers to emphasize in some plots.
LAYER_SAMPLE_STRIDE = 1
TOP_LAYER_K = 6
N_NEIGHBORS = 8

# Visualization settings.
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for p in [OUT_DIR, CSV_DIR, PLOTS_DIR, REPORTS_DIR, CACHE_DIR, NPZ_DIR]:
    os.makedirs(p, exist_ok=True)

TOKENIZER = None
MODEL = None
BLOCKS = None
FINAL_NORM = None
LM_HEAD = None
MODEL_LAYERS = None
MODEL_HIDDEN = None

# ============================================================
# UTILS
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)


def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def preview_text(s, max_len=160):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")


def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)


def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass


def safe_mean(values):
    s = pd.Series(values)
    return float(s.mean()) if len(s) else float("nan")


def safe_std(values):
    s = pd.Series(values)
    return float(s.std()) if len(s) else float("nan")


def safe_pearson(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if len(x) < 2 or np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def row_norm(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / (n + eps)


def center(x):
    x = np.asarray(x, dtype=np.float64)
    return x - x.mean(axis=0, keepdims=True)


def cosine_matrix(a, b):
    a = row_norm(a)
    b = row_norm(b)
    return a @ b.T


def cka_linear(X, Y):
    """
    Feature-space Linear CKA: compares the feature covariance structure of X and Y.
    Handles unequal sample sizes by computing the inner product of expected HxH covariance matrices.
    """
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if len(X) < 2 or len(Y) < 2:
        return float("nan")

    X = center(X)
    Y = center(Y)

    # Compute expected feature covariance matrices
    Cx = (X.T @ X) / len(X)
    Cy = (Y.T @ Y) / len(Y)

    # Frobenius inner product: trace(Cx^T Cy) = sum(Cx * Cy)
    num = np.sum(Cx * Cy)
    den = math.sqrt(np.sum(Cx * Cx) * np.sum(Cy * Cy))

    if den <= 1e-12:
        return float("nan")
    return float(num / den)


def principal_angles_overlap(A, B, k=8):
    """Subspace overlap score via principal components."""
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    if len(A) < 2 or len(B) < 2:
        return float("nan")
    
    k = max(1, min(k, A.shape[1], B.shape[1], len(A) - 1, len(B) - 1))
    if k < 1:
        return float("nan")
        
    pca_a = PCA(n_components=k, random_state=SEED)
    pca_b = PCA(n_components=k, random_state=SEED)
    pca_a.fit(center(A))
    pca_b.fit(center(B))
    Ua = pca_a.components_.T
    Ub = pca_b.components_.T
    M = Ua.T @ Ub
    score = np.linalg.norm(M, ord="fro") / math.sqrt(k)
    return float(np.clip(score, 0.0, 1.0))


def pairwise_cosine_mean(X, Y):
    C = cosine_matrix(X, Y)
    return float(np.mean(C)) if C.size else float("nan")


def token_similarity(a, b):
    sa = set(re.findall(r"\w+", str(a).lower()))
    sb = set(re.findall(r"\w+", str(b).lower()))
    if not sa and not sb:
        return 1.0
    if not sa or not sb:
        return 0.0
    return float(len(sa & sb) / len(sa | sb))


def bar_plot(labels, values, title, path, ylabel="Value", rotate=30):
    plt.figure(figsize=(10.5, 4.8))
    ax = plt.gca()
    colors = plt.get_cmap("Set2")(np.linspace(0.12, 0.88, len(labels)))
    ax.bar(labels, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotate)
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h):
            ax.annotate(f"{h:.3f}", (p.get_x() + p.get_width() / 2.0, h), ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis", annotate=False, fmt="{:.2f}"):
    plt.figure(figsize=(max(9, len(xlabels) * 0.55), max(6, len(ylabels) * 0.38)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    if annotate:
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if np.isfinite(mat[i, j]):
                    plt.text(j, i, fmt.format(mat[i, j]), ha="center", va="center", fontsize=7,
                             color="white" if abs(mat[i, j]) > 0.5 else "black")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def scatter(x, y, title, path, xlabel="X", ylabel="Y", color=None, alpha=0.8):
    plt.figure(figsize=(7.2, 5.8))
    plt.scatter(x, y, c=color, alpha=alpha, s=24)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def ridgeline_plot(distributions, labels, title, path, xlim=None):
    plt.figure(figsize=(11.0, 7.8))
    xs = np.linspace(0, 1, 400)
    cmap = plt.get_cmap("viridis")
    for i, (dist, label) in enumerate(zip(distributions, labels)):
        dist = np.asarray(dist, dtype=np.float64)
        if len(dist) == 0:
            continue
        hist, bins = np.histogram(dist, bins=60, range=(0, 1), density=True)
        centers = (bins[:-1] + bins[1:]) / 2.0
        curve = np.interp(xs, centers, hist)
        kernel = np.exp(-0.5 * ((np.arange(-10, 11)) / 3.0) ** 2)
        kernel /= kernel.sum()
        curve = np.convolve(curve, kernel, mode="same")
        y = (curve / (curve.max() + 1e-12)) * 0.75
        offset = i * 1.15
        plt.fill_between(xs, offset, offset + y, color=cmap(i / max(1, len(distributions) - 1)), alpha=0.88)
        plt.plot(xs, offset + y, color="white", linewidth=1.0)
        plt.text(1.01, offset + 0.18, label, fontsize=9, va="center")
    plt.title(title)
    plt.yticks([])
    plt.xlabel("Cosine / overlap score")
    if xlim is not None:
        plt.xlim(*xlim)
    else:
        plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def line_with_points(x, ys, labels, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(10.5, 5.4))
    for y, label in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=2, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_3d_task_manifold(X, task_labels, title, path):
    pca = PCA(n_components=3, random_state=SEED)
    emb3d = pca.fit_transform(StandardScaler().fit_transform(X))
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    tasks = sorted(set(task_labels))
    cmap = plt.get_cmap("tab10")
    for i, t in enumerate(tasks):
        mask = np.array(task_labels) == t
        ax.scatter(emb3d[mask, 0], emb3d[mask, 1], emb3d[mask, 2], label=t, color=cmap(i), alpha=0.8, s=25)
    ax.set_title(title)
    ax.set_xlabel("PCA-1")
    ax.set_ylabel("PCA-2")
    ax.set_zlabel("PCA-3")
    ax.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def violin_plot_metric(df, metric, task_col, title, path, ylabel="Value"):
    plt.figure(figsize=(10.5, 6))
    tasks = sorted(df[task_col].unique())
    data = [df[df[task_col] == t][metric].dropna().to_numpy(dtype=np.float64) for t in tasks]
    valid_idx = [i for i, d in enumerate(data) if len(d) > 0]
    
    if valid_idx:
        parts = plt.violinplot([data[i] for i in valid_idx], showmeans=True)
        for pc in parts['bodies']:
            pc.set_facecolor('tab:purple')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.xticks(range(1, len(valid_idx) + 1), [tasks[i] for i in valid_idx], rotation=25, ha="right")
        
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_correlation_matrix(df, cols, title, path):
    sub_df = df[cols].dropna()
    if len(sub_df) < 2:
        return
    corr = sub_df.corr().values
    heatmap(corr, cols, cols, title, path, cmap="coolwarm", annotate=True, fmt="{:.2f}")


def plot_task_dendrogram(centroids, path):
    if not HAS_SCIPY:
        return
    tasks = list(centroids.keys())
    # Grab final layer centroids for clustering
    X = np.array([centroids[t][-1] for t in tasks])
    Z = linkage(X, method='average', metric='cosine')
    plt.figure(figsize=(10, 6))
    dendrogram(Z, labels=tasks, orientation='right', color_threshold=0.7*max(Z[:,2]))
    plt.title("Task Taxonomy Tree (Hierarchical Clustering on Final Layer)")
    plt.xlabel("Cosine Distance")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def quiver_plot(XY, labels, task_names, title, path):
    plt.figure(figsize=(10.0, 7.0))
    cmap = plt.get_cmap("tab10")
    for i, task in enumerate(task_names):
        pts = XY[labels == task]
        if len(pts) == 0:
            continue
        plt.scatter(pts[:, 0], pts[:, 1], s=28, alpha=0.75, color=cmap(i), label=task)
    plt.title(title)
    plt.xlabel("Projection-1")
    plt.ylabel("Projection-2")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


# ============================================================
# SAMPLING
# ============================================================

def cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx


def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cached_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

# ============================================================
# LOAD MODEL
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (HAS_PEFT and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER


def discover_blocks(model):
    candidates = [
        ("model.layers", getattr(getattr(model, "model", None), "layers", None)),
        ("model.decoder.layers", getattr(getattr(getattr(model, "model", None), "decoder", None), "layers", None)),
        ("transformer.h", getattr(getattr(model, "transformer", None), "h", None)),
        ("gpt_neox.layers", getattr(getattr(model, "gpt_neox", None), "layers", None)),
        ("decoder.layers", getattr(getattr(model, "decoder", None), "layers", None)),
    ]
    for name, maybe_blocks in candidates:
        if maybe_blocks is not None:
            maybe_blocks = list(maybe_blocks)
            if len(maybe_blocks) > 0:
                return maybe_blocks, name
    raise RuntimeError("Could not locate transformer blocks.")


def discover_final_norm(model):
    candidates = [
        ("model.norm", getattr(getattr(model, "model", None), "norm", None)),
        ("transformer.ln_f", getattr(getattr(model, "transformer", None), "ln_f", None)),
        ("gpt_neox.final_layer_norm", getattr(getattr(model, "gpt_neox", None), "final_layer_norm", None)),
        ("decoder.final_layer_norm", getattr(getattr(model, "decoder", None), "final_layer_norm", None)),
    ]
    for name, mod in candidates:
        if mod is not None:
            return mod, name
    return None, None


def discover_lm_head(model):
    if hasattr(model, "lm_head"):
        return model.lm_head, "lm_head"
    if hasattr(model, "embed_out"):
        return model.embed_out, "embed_out"
    raise RuntimeError("Could not locate lm_head / embed_out.")


def load_model():
    global MODEL, BLOCKS, FINAL_NORM, LM_HEAD, MODEL_LAYERS, MODEL_HIDDEN
    load_tokenizer()
    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if HAS_PEFT and os.path.exists(SFT_PATH):
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except Exception:
            pass

    try:
        model.config.use_cache = False
    except Exception:
        pass

    blocks, block_name = discover_blocks(model)
    final_norm, norm_name = discover_final_norm(model)
    lm_head, head_name = discover_lm_head(model)

    MODEL = model
    BLOCKS = blocks
    FINAL_NORM = final_norm
    LM_HEAD = lm_head
    MODEL_LAYERS = len(BLOCKS)
    try:
        MODEL_HIDDEN = int(model.config.hidden_size)
    except Exception:
        MODEL_HIDDEN = int(lm_head.weight.shape[1])

    print(f"Loaded model with {MODEL_LAYERS} blocks ({block_name}), norm={norm_name}, head={head_name}")
    return model, TOKENIZER

# ============================================================
# DATA LOADING
# ============================================================

def load_gsm8k_samples(n=N_GSM8K):
    ds = load_dataset("gsm8k", "main", split="test")
    ds, idx = sample_dataset(ds, "gsm8k_geometry_atlas", n)
    samples = []
    for i, s in enumerate(ds):
        _, gold = s["answer"].split("####", 1)
        samples.append({
            "task": "gsm8k",
            "sample_id": i,
            "source_index": idx[i],
            "subject": "gsm8k",
            "question": s["question"],
            "gold": gold.strip(),
            "gold_text": s["answer"],
        })
    return samples


def load_strategyqa_samples(n=N_STRATEGYQA):
    ds = load_dataset("ChilleD/StrategyQA", split="test")
    ds, idx = sample_dataset(ds, "strategyqa_geometry_atlas", n)
    samples = []
    for i, s in enumerate(ds):
        samples.append({
            "task": "strategyqa",
            "sample_id": i,
            "source_index": idx[i],
            "subject": "strategyqa",
            "question": s["question"],
            "gold": "yes" if bool(s["answer"]) else "no",
        })
    return samples


def load_mmlu_samples(subjects=MMLU_SUBJECTS, n_per_subject=N_MMLU_PER_SUBJECT):
    all_samples = []
    for subject in subjects:
        ds = load_dataset("cais/mmlu", subject, split="test")
        ds, idx = sample_dataset(ds, f"mmlu_geometry_atlas_{subject}", n_per_subject)
        for i, s in enumerate(ds):
            all_samples.append({
                "task": "mmlu",
                "sample_id": len(all_samples),
                "source_index": idx[i],
                "subject": subject,
                "question": s["question"],
                "choices": list(s["choices"]),
                "gold": chr(65 + int(s["answer"])),
            })
    return all_samples

# ============================================================
# PROMPTS / TARGETS
# ============================================================

def build_prompt(sample):
    if sample["task"] == "gsm8k":
        return f"Question: {sample['question']}\nAnswer:\n"
    if sample["task"] == "strategyqa":
        return f"Question: {sample['question']}\nAnswer only yes or no:\n"
    if sample["task"] == "mmlu":
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
        return f"Question: {sample['question']}\n{opts}\nAnswer ONLY with a single letter (A, B, C, or D):\n"
    raise ValueError(sample["task"])


def target_text(sample):
    if sample["task"] == "gsm8k":
        return sample["gold"]
    if sample["task"] == "strategyqa":
        return sample["gold"]
    if sample["task"] == "mmlu":
        return sample["gold"]
    raise ValueError(sample["task"])


def parse_prediction(sample, completion):
    if sample["task"] == "gsm8k":
        nums = re.findall(r"-?\d+\.?\d*", completion.replace(",", ""))
        return nums[-1] if nums else None
    if sample["task"] == "strategyqa":
        return extract_yes_no(completion)
    if sample["task"] == "mmlu":
        pred = extract_mcq(completion)
        return pred
    return None


def extract_yes_no(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None


def extract_mcq(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span_up = span.upper().strip()
    if span_up in ("A", "B", "C", "D"):
        return span_up
    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"<ANSWER>\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
    ]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits:
            return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", span_up)
    return hits[-1].upper() if hits else None


def canonical_correct(sample, pred):
    if sample["task"] == "gsm8k":
        try:
            return abs(float(pred) - float(sample["gold"])) <= 1e-6
        except Exception:
            return False
    if sample["task"] in ("strategyqa", "mmlu"):
        return str(pred).strip().upper() == str(sample["gold"]).strip().upper()
    return False

# ============================================================
# MODEL EXECUTION / FEATURE EXTRACTION
# ============================================================

def token_ids_for_text(tokenizer, text):
    forms = [
        text, f" {text}", f"\n{text}", text.upper(), f" {text.upper()}", text.lower(), f" {text.lower()}"
    ]
    ids = []
    for form in forms:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if len(enc) == 1:
            ids.append(enc[0])
    if not ids:
        enc = tokenizer.encode(text, add_special_tokens=False)
        if len(enc) > 0:
            ids.append(enc[0])
    return sorted(set(ids))


def target_from_ids(logits, ids):
    if len(ids) == 0:
        return None, float("nan")
    vals = [float(logits[i].item()) for i in ids]
    idx = int(np.argmax(vals))
    return ids[idx], vals[idx]


@torch.inference_mode()
def generate_completion(prompt, max_new_tokens=64):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    out = MODEL.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=TOKENIZER.eos_token_id,
        eos_token_id=TOKENIZER.eos_token_id,
    )
    full_output = TOKENIZER.decode(out[0], skip_special_tokens=True)
    prompt_len = inputs["input_ids"].shape[1]
    completion = TOKENIZER.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion


def pooled_hidden_states(outputs, attention_mask, pooling="last"):
    hs = []
    mask = attention_mask.float()
    lengths = mask.sum(dim=1).long().clamp(min=1)
    for h in outputs.hidden_states:
        if pooling == "last":
            idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, h.shape[-1])
            pooled = h.gather(1, idx).squeeze(1)
        elif pooling == "mean":
            pooled = (h * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1.0)
        else:
            raise ValueError(pooling)
        if FINAL_NORM is not None:
            try:
                pooled = FINAL_NORM(pooled)
            except Exception:
                pass
        hs.append(pooled.detach().float().cpu().numpy())
    return hs


def analyze_sample(sample):
    prompt = build_prompt(sample)
    target = target_text(sample)
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        outputs = MODEL(
            **inputs,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )
        full_output, completion = generate_completion(prompt, max_new_tokens=80)

    pred = parse_prediction(sample, completion)
    correct = canonical_correct(sample, pred)
    pooled_layers = pooled_hidden_states(outputs, inputs["attention_mask"], pooling="last")
    logits = outputs.logits[0, -1].float()
    log_probs = torch.log_softmax(logits, dim=-1)
    probs = torch.softmax(logits, dim=-1)

    if sample["task"] == "strategyqa":
        target_ids = token_ids_for_text(TOKENIZER, sample["gold"])
    elif sample["task"] == "mmlu":
        target_ids = token_ids_for_text(TOKENIZER, sample["gold"])
    else:
        target_ids = token_ids_for_text(TOKENIZER, sample["gold"])
        
    tid, tlogit = target_from_ids(logits, target_ids)
    if tid is None:
        tid = int(TOKENIZER.eos_token_id)
        tlogit = float(logits[tid].item())
        
    target_prob = float(probs[tid].item())
    target_logprob = float(log_probs[tid].item())
    masked = logits.clone()
    masked[target_ids] = float("-inf")
    target_margin = float(tlogit - masked.max().item())
    target_rank = int((logits > tlogit).sum().item() + 1)

    top_probs, top_ids = torch.topk(probs, k=min(6, probs.shape[-1]))
    top_tokens = TOKENIZER.convert_ids_to_tokens(top_ids.tolist())
    top_snapshot = [{"token": t, "id": int(i), "prob": float(p)} for t, i, p in zip(top_tokens, top_ids.tolist(), top_probs.tolist())]

    return {
        "prompt": prompt,
        "full_output": full_output,
        "completion": completion,
        "prediction": pred,
        "correct": bool(correct),
        "target_token_ids": to_jsonable(target_ids),
        "target_id": int(tid),
        "target_logit": float(tlogit),
        "target_logprob": float(target_logprob),
        "target_prob": float(target_prob),
        "target_margin": float(target_margin),
        "target_rank": int(target_rank),
        "top_snapshot": to_jsonable(top_snapshot),
        "pooled_layers": [x.squeeze(0).tolist() for x in pooled_layers],
        "prompt_tokens": int(inputs["input_ids"].shape[1]),
        "completion_tokens": len(TOKENIZER.encode(completion, add_special_tokens=False)),
    }

# ============================================================
# DATASET HELPERS
# ============================================================

def load_all_samples():
    samples = []
    samples.extend(load_gsm8k_samples())
    samples.extend(load_strategyqa_samples())
    samples.extend(load_mmlu_samples())
    return samples

# ============================================================
# MAIN ANALYSIS
# ============================================================

def build_feature_table(samples):
    rows = []
    print("Extracting layerwise hidden states and predictions ...")
    for i, sample in enumerate(samples):
        print(f"Sample {i+1}/{len(samples)} | {sample['task']} | {sample.get('subject', sample['task'])} | {preview_text(sample['question'], 90)}")
        result = analyze_sample(sample)
        rows.append({
            **sample,
            **result,
        })
        free_memory()
    df = pd.DataFrame(rows)
    return df


def layer_matrix_from_df(df, layer_col="pooled_layers"):
    arr = np.array(df[layer_col].tolist(), dtype=np.float32)
    return arr


def compute_task_centroids(arr, task_labels):
    tasks = sorted(set(task_labels))
    centroids = {}
    for t in tasks:
        centroids[t] = arr[np.array(task_labels) == t].mean(axis=0)
    return centroids


def compute_layerwise_task_centroids(arr, task_labels):
    tasks = sorted(set(task_labels))
    out = {}
    for t in tasks:
        mask = np.array(task_labels) == t
        out[t] = arr[mask].mean(axis=0)
    return out


def trajectory_distances(centroids):
    tasks = list(centroids.keys())
    L = centroids[tasks[0]].shape[0]
    rows = []
    for layer in range(L):
        for a, b in combinations(tasks, 2):
            da = centroids[a][layer]
            db = centroids[b][layer]
            rows.append({
                "layer": layer,
                "task_a": a,
                "task_b": b,
                "cosine": float(np.dot(da, db) / ((np.linalg.norm(da) * np.linalg.norm(db)) + 1e-12)),
                "l2": float(np.linalg.norm(da - db)),
            })
    return pd.DataFrame(rows)


def cka_matrix_by_layer(arr, task_labels):
    tasks = sorted(set(task_labels))
    L = arr.shape[1]
    mats = []
    for layer in range(L):
        layer_rows = []
        for a in tasks:
            Xa = arr[np.array(task_labels) == a, layer, :]
            layer_rows.append(Xa)
        cka = np.zeros((len(tasks), len(tasks)), dtype=np.float64)
        for i, a in enumerate(tasks):
            for j, b in enumerate(tasks):
                if i == j:
                    cka[i, j] = 1.0
                else:
                    cka[i, j] = cka_linear(layer_rows[i], layer_rows[j])
        mats.append(cka)
    return tasks, np.stack(mats, axis=0)


def subspace_overlap_by_layer(arr, task_labels, k=8):
    tasks = sorted(set(task_labels))
    L = arr.shape[1]
    mats = []
    for layer in range(L):
        layer_rows = [arr[np.array(task_labels) == t, layer, :] for t in tasks]
        mat = np.zeros((len(tasks), len(tasks)), dtype=np.float64)
        for i in range(len(tasks)):
            for j in range(len(tasks)):
                if i == j:
                    mat[i, j] = 1.0
                else:
                    mat[i, j] = principal_angles_overlap(layer_rows[i], layer_rows[j], k=k)
        mats.append(mat)
    return tasks, np.stack(mats, axis=0)


def principal_direction_drift(centroids):
    tasks = list(centroids.keys())
    L = centroids[tasks[0]].shape[0]
    rows = []
    for t in tasks:
        X = centroids[t]
        pca = PCA(n_components=min(3, X.shape[0], X.shape[1]), random_state=SEED)
        pca.fit(center(X))
        for layer in range(1, L):
            prev = centroids[t][layer - 1]
            curr = centroids[t][layer]
            drift = float(np.dot(prev, curr) / ((np.linalg.norm(prev) * np.linalg.norm(curr)) + 1e-12))
            rows.append({
                "task": t,
                "layer": layer,
                "drift_cosine": drift,
                "explained_var1": float(pca.explained_variance_ratio_[0]),
            })
    return pd.DataFrame(rows)


def reduce_2d(X, method="pca"):
    X = np.asarray(X, dtype=np.float64)
    Xs = StandardScaler().fit_transform(X)
    n_samples = len(Xs)
    
    if method == "umap" and HAS_UMAP:
        n_neighbors = min(15, n_samples - 2) if n_samples > 2 else 2
        if n_neighbors >= 2:
            reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=n_neighbors, min_dist=0.2)
            return reducer.fit_transform(Xs)
        else:
            method = "pca" 

    if method == "tsne" and HAS_TSNE:
        perplexity = min(30, max(5, n_samples // 3))
        if n_samples > 5:
            reducer = TSNE(n_components=2, random_state=SEED, perplexity=perplexity)
            return reducer.fit_transform(Xs)
        else:
            method = "pca" 

    pca = PCA(n_components=2, random_state=SEED)
    return pca.fit_transform(Xs)


def plot_2d_task_manifold(emb2d, task_labels, title, path):
    plt.figure(figsize=(9.5, 7.0))
    tasks = sorted(set(task_labels))
    cmap = plt.get_cmap("tab10")
    for i, t in enumerate(tasks):
        mask = np.array(task_labels) == t
        plt.scatter(emb2d[mask, 0], emb2d[mask, 1], s=26, alpha=0.75, label=t, color=cmap(i))
    plt.title(title)
    plt.xlabel("Dim-1")
    plt.ylabel("Dim-2")
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_layer_trajectory(centroids, path, tasks_to_plot=None):
    tasks = list(centroids.keys())
    if tasks_to_plot is None:
        tasks_to_plot = tasks
    fig, ax = plt.subplots(figsize=(9.5, 7.2))
    cmap = plt.get_cmap("tab10")
    for i, t in enumerate(tasks_to_plot):
        C = centroids[t]
        emb2d = reduce_2d(C, method="pca")
        ax.plot(emb2d[:, 0], emb2d[:, 1], marker="o", linewidth=2, label=t, color=cmap(i))
        for l in range(0, len(emb2d), max(1, len(emb2d) // 6)):
            ax.annotate(str(l), (emb2d[l, 0], emb2d[l, 1]), fontsize=8, xytext=(3, 3), textcoords="offset points")
    ax.set_title("Layerwise task centroid trajectories")
    ax.set_xlabel("PCA-1")
    ax.set_ylabel("PCA-2")
    ax.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_layerwise_embedding_cloud(arr, task_labels, path, layer_idx=-1):
    X = arr[:, layer_idx, :]
    emb = reduce_2d(X, method="umap" if HAS_UMAP else "pca")
    plot_2d_task_manifold(emb, task_labels, f"Layer {layer_idx} embedding cloud", path)


def plot_centroid_separation(centroids, path):
    tasks = list(centroids.keys())
    mat = np.zeros((len(tasks), len(tasks)), dtype=np.float64)
    last = [centroids[t][-1] for t in tasks]
    for i in range(len(tasks)):
        for j in range(len(tasks)):
            if i == j:
                mat[i, j] = 0.0
            else:
                mat[i, j] = float(np.linalg.norm(last[i] - last[j]) / (np.linalg.norm(last[i]) + np.linalg.norm(last[j]) + 1e-12))
    heatmap(mat, tasks, tasks, "Final-layer centroid separation", path, xlabel="Task", ylabel="Task", cmap="coolwarm", annotate=True, fmt="{:.2f}")


def plot_cka_layers(tasks, cka_stack, path):
    L, T, _ = cka_stack.shape
    fig, axes = plt.subplots(1, min(3, L), figsize=(18, 5.3), constrained_layout=True)
    if len(np.atleast_1d(axes).shape) == 0:
        axes = [axes]
    layers_to_show = [0, L // 2, L - 1] if L >= 3 else list(range(L))
    for ax, layer in zip(np.atleast_1d(axes), layers_to_show):
        im = ax.imshow(cka_stack[layer], cmap="viridis", vmin=0.0, vmax=1.0)
        ax.set_title(f"CKA matrix layer {layer}")
        ax.set_xticks(range(T))
        ax.set_yticks(range(T))
        ax.set_xticklabels(tasks, rotation=35, ha="right")
        ax.set_yticklabels(tasks)
        for i in range(T):
            for j in range(T):
                ax.text(j, i, f"{cka_stack[layer, i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if cka_stack[layer, i, j] > 0.55 else "black")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.suptitle("Representational similarity (CKA) across tasks", fontsize=16, y=1.03)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()


def plot_subspace_ribbon(tasks, overlap_stack, path):
    L = overlap_stack.shape[0]
    pairs = list(combinations(range(len(tasks)), 2))
    fig, ax = plt.subplots(figsize=(11.2, 6.6))
    for i, (a, b) in enumerate(pairs):
        y = overlap_stack[:, a, b]
        ax.plot(range(L), y, marker="o", linewidth=2, label=f"{tasks[a]} ↔ {tasks[b]}")
    ax.set_title("Layerwise subspace overlap ribbon")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Subspace overlap")
    ax.legend(frameon=True, fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def plot_drift_dashboard(drift_df, path):
    fig, axes = plt.subplots(1, 2, figsize=(14.0, 5.2), constrained_layout=True)
    for ax, metric, title in [
        (axes[0], "drift_cosine", "Principal direction drift cosine"),
        (axes[1], "explained_var1", "First principal component variance"),
    ]:
        for t, g in drift_df.groupby("task"):
            ax.plot(g["layer"], g[metric], marker="o", linewidth=2, label=t)
        ax.set_title(title)
        ax.set_xlabel("Layer")
        ax.set_ylabel(metric)
        ax.legend(frameon=True, fontsize=8)
    plt.savefig(path, dpi=180)
    plt.close()


def plot_task_geometry_dashboard(centroids, tasks, cka_stack, overlap_stack, emb2d, task_labels, path):
    fig, axes = plt.subplots(2, 3, figsize=(20, 12), constrained_layout=True)

    # Panel 1: 2D manifold
    cmap = plt.get_cmap("tab10")
    for i, t in enumerate(tasks):
        mask = np.array(task_labels) == t
        axes[0, 0].scatter(emb2d[mask, 0], emb2d[mask, 1], s=24, alpha=0.75, color=cmap(i), label=t)
    axes[0, 0].set_title("Task manifold projection")
    axes[0, 0].set_xlabel("Dim-1")
    axes[0, 0].set_ylabel("Dim-2")
    axes[0, 0].legend(frameon=True, fontsize=8)

    # Panel 2: centroid trajectories.
    for i, t in enumerate(tasks):
        C = centroids[t]
        C2 = reduce_2d(C, method="pca")
        axes[0, 1].plot(C2[:, 0], C2[:, 1], marker="o", linewidth=2, label=t, color=cmap(i))
        for l in range(0, len(C2), max(1, len(C2) // 5)):
            axes[0, 1].annotate(str(l), (C2[l, 0], C2[l, 1]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    axes[0, 1].set_title("Layerwise centroid trajectories")
    axes[0, 1].set_xlabel("PCA-1")
    axes[0, 1].set_ylabel("PCA-2")
    axes[0, 1].legend(frameon=True, fontsize=8)

    # Panel 3: final centroid separation heatmap.
    last = [centroids[t][-1] for t in tasks]
    mat = np.zeros((len(tasks), len(tasks)), dtype=np.float64)
    for i in range(len(tasks)):
        for j in range(len(tasks)):
            if i == j:
                mat[i, j] = 0.0
            else:
                mat[i, j] = float(np.linalg.norm(last[i] - last[j]) / (np.linalg.norm(last[i]) + np.linalg.norm(last[j]) + 1e-12))
    im = axes[0, 2].imshow(mat, cmap="coolwarm")
    axes[0, 2].set_title("Final centroid separation")
    axes[0, 2].set_xticks(range(len(tasks)))
    axes[0, 2].set_yticks(range(len(tasks)))
    axes[0, 2].set_xticklabels(tasks, rotation=30, ha="right")
    axes[0, 2].set_yticklabels(tasks)
    fig.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

    # Panel 4: CKA heatmap at final layer.
    im2 = axes[1, 0].imshow(cka_stack[-1], cmap="viridis", vmin=0.0, vmax=1.0)
    axes[1, 0].set_title("Final-layer CKA")
    axes[1, 0].set_xticks(range(len(tasks)))
    axes[1, 0].set_yticks(range(len(tasks)))
    axes[1, 0].set_xticklabels(tasks, rotation=30, ha="right")
    axes[1, 0].set_yticklabels(tasks)
    fig.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

    # Panel 5: overlap ribbon summary.
    pairs = list(combinations(range(len(tasks)), 2))
    for idx, (a, b) in enumerate(pairs):
        axes[1, 1].plot(range(cka_stack.shape[0]), overlap_stack[:, a, b], marker="o", linewidth=2, label=f"{tasks[a]}↔{tasks[b]}")
    axes[1, 1].set_title("Layerwise subspace overlap")
    axes[1, 1].set_xlabel("Layer")
    axes[1, 1].set_ylabel("Overlap")
    axes[1, 1].legend(frameon=True, fontsize=8)

    # Panel 6: centroid distance trajectory (mean pairwise distance).
    mean_pairwise = []
    for layer in range(centroids[tasks[0]].shape[0]):
        dsum = 0.0
        cnt = 0
        for i in range(len(tasks)):
            for j in range(i + 1, len(tasks)):
                dsum += np.linalg.norm(centroids[tasks[i]][layer] - centroids[tasks[j]][layer])
                cnt += 1
        mean_pairwise.append(dsum / max(1, cnt))
    axes[1, 2].plot(range(len(mean_pairwise)), mean_pairwise, marker="o", linewidth=2)
    axes[1, 2].set_title("Mean pairwise centroid distance")
    axes[1, 2].set_xlabel("Layer")
    axes[1, 2].set_ylabel("Distance")

    plt.suptitle("Task geometry atlas dashboard", fontsize=16, y=1.01)
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading model and tokenizer ...")
    load_model()

    print("Loading datasets ...")
    gsm = load_gsm8k_samples()
    qa = load_strategyqa_samples()
    mmlu = load_mmlu_samples()
    samples = gsm + qa + mmlu

    print(f"Loaded {len(gsm)} GSM8K, {len(qa)} StrategyQA, {len(mmlu)} MMLU samples")

    rows = []
    for i, sample in enumerate(samples):
        print(f"Sample {i+1}/{len(samples)} | {sample['task']} | {sample.get('subject', '')} | {preview_text(sample['question'], 85)}")
        out = analyze_sample(sample)
        rows.append({**sample, **out})
        free_memory()

    df = pd.DataFrame(rows)
    save_df(df, os.path.join(CSV_DIR, "sample_layerwise_features.csv"))

    # Stack layerwise pooled embeddings.
    arr = np.array(df["pooled_layers"].tolist(), dtype=np.float32)
    task_labels = df["task"].tolist()
    subject_labels = df["subject"].tolist()

    # Metadata for each layer.
    L = arr.shape[1]
    H = arr.shape[2]
    meta = {
        "n_samples": int(arr.shape[0]),
        "n_layers": int(L),
        "hidden_size": int(H),
        "tasks": sorted(df["task"].unique().tolist()),
        "subjects": sorted(df["subject"].unique().tolist()),
        "model_layers": MODEL_LAYERS,
        "model_hidden": MODEL_HIDDEN,
    }
    save_json(meta, os.path.join(REPORTS_DIR, "metadata.json"))

    # Save raw array for later.
    if len(arr) > 0:
        np.savez_compressed(
            os.path.join(NPZ_DIR, "layerwise_embeddings.npz"),
            embeddings=arr,
            task_labels=np.array(task_labels),
            subject_labels=np.array(subject_labels),
            question_id=np.array(df["sample_id"].tolist()),
            question=np.array(df["question"].tolist()),
            gold=np.array(df["gold"].tolist()),
            prediction=np.array(df["prediction"].tolist()),
            correct=np.array(df["correct"].tolist()),
        )

    # Task centroids per layer.
    centroids = compute_layerwise_task_centroids(arr, task_labels)
    centroid_rows = []
    for t, C in centroids.items():
        for layer in range(C.shape[0]):
            centroid_rows.append({
                "task": t,
                "layer": layer,
                "centroid": to_jsonable(C[layer].tolist()),
            })
    save_df(pd.DataFrame(centroid_rows), os.path.join(CSV_DIR, "task_centroids_by_layer.csv"))

    # Layerwise task distance metrics.
    pair_df = trajectory_distances(centroids)
    save_df(pair_df, os.path.join(CSV_DIR, "task_pairwise_layer_distances.csv"))

    # CKA and subspace overlap.
    tasks, cka_stack = cka_matrix_by_layer(arr, task_labels)
    _, overlap_stack = subspace_overlap_by_layer(arr, task_labels, k=8)
    np.savez_compressed(
        os.path.join(NPZ_DIR, "geometry_matrices.npz"),
        cka_stack=cka_stack,
        overlap_stack=overlap_stack,
        tasks=np.array(tasks),
    )

    # Principal direction drift.
    drift_rows = []
    for t, C in centroids.items():
        pca = PCA(n_components=min(3, C.shape[0], C.shape[1]), random_state=SEED)
        pca.fit(center(C))
        for layer in range(1, C.shape[0]):
            prev = C[layer - 1]
            curr = C[layer]
            drift_rows.append({
                "task": t,
                "layer": layer,
                "drift_cosine": float(np.dot(prev, curr) / ((np.linalg.norm(prev) * np.linalg.norm(curr)) + 1e-12)),
                "step_distance": float(np.linalg.norm(curr - prev)),
                "explained_var1": float(pca.explained_variance_ratio_[0]) if len(pca.explained_variance_ratio_) > 0 else float("nan"),
            })
    drift_df = pd.DataFrame(drift_rows)
    save_df(drift_df, os.path.join(CSV_DIR, "principal_direction_drift.csv"))

    # Global summaries.
    task_summary = df.groupby("task", as_index=False).agg(
        n_samples=("sample_id", "count"),
        accuracy=("correct", "mean"),
        mean_prompt_tokens=("prompt_tokens", "mean"),
        mean_completion_tokens=("completion_tokens", "mean"),
        mean_target_margin=("target_margin", "mean"),
        mean_target_prob=("target_prob", "mean"),
        mean_target_rank=("target_rank", "mean"),
    )
    subject_summary = df.groupby("subject", as_index=False).agg(
        n_samples=("sample_id", "count"),
        accuracy=("correct", "mean"),
        mean_prompt_tokens=("prompt_tokens", "mean"),
        mean_completion_tokens=("completion_tokens", "mean"),
        mean_target_margin=("target_margin", "mean"),
        mean_target_prob=("target_prob", "mean"),
        mean_target_rank=("target_rank", "mean"),
    )
    save_df(task_summary, os.path.join(CSV_DIR, "task_summary.csv"))
    save_df(subject_summary, os.path.join(CSV_DIR, "subject_summary.csv"))

    # Reduce a single view of all final-layer embeddings.
    X_final = arr[:, -1, :]
    emb2d = reduce_2d(X_final, method="umap" if HAS_UMAP else ("tsne" if HAS_TSNE else "pca"))
    if emb2d is None or len(emb2d) == 0:
        emb2d = PCA(n_components=2, random_state=SEED).fit_transform(StandardScaler().fit_transform(X_final))
    plot_2d_task_manifold(emb2d, np.array(task_labels), "Final-layer task manifold", os.path.join(PLOTS_DIR, "task_manifold_final_layer.png"))

    # --- NEW VISUALIZATIONS ---
    plot_3d_task_manifold(X_final, np.array(task_labels), "3D Final-layer task manifold", os.path.join(PLOTS_DIR, "task_manifold_3d_final.png"))
    plot_task_dendrogram(centroids, os.path.join(PLOTS_DIR, "task_taxonomy_dendrogram.png"))
    violin_plot_metric(df, "target_margin", "task", "Target Margin Distribution by Task", os.path.join(PLOTS_DIR, "violin_target_margin.png"), "Target Margin")
    violin_plot_metric(df, "target_prob", "task", "Target Probability Distribution by Task", os.path.join(PLOTS_DIR, "violin_target_prob.png"), "Target Probability")
    plot_correlation_matrix(df, ["prompt_tokens", "completion_tokens", "target_margin", "target_prob", "target_rank"], "Sample Feature Correlation", os.path.join(PLOTS_DIR, "sample_feature_correlation.png"))
    # --------------------------

    # Layerwise centroid trajectories.
    plot_layer_trajectory(centroids, os.path.join(PLOTS_DIR, "task_centroid_trajectories.png"), tasks_to_plot=sorted(centroids.keys()))

    # Layerwise embedding cloud on final layer.
    plot_layer_trajectory(centroids, os.path.join(PLOTS_DIR, "task_centroid_trajectory_pca.png"))
    plot_layerwise_embedding_cloud(arr, np.array(task_labels), os.path.join(PLOTS_DIR, "final_layer_embedding_cloud.png"), layer_idx=-1)

    # Centroid separation.
    plot_centroid_separation(centroids, os.path.join(PLOTS_DIR, "final_centroid_separation_heatmap.png"))

    # CKA heatmaps.
    plot_cka_layers(tasks, cka_stack, os.path.join(PLOTS_DIR, "cka_layers_dashboard.png"))
    tasks_idx = {t: i for i, t in enumerate(tasks)}
    pair_names = [f"{a}↔{b}" for a, b in combinations(tasks, 2)]
    pair_mat = np.zeros((cka_stack.shape[0], len(pair_names)), dtype=np.float64)
    for li in range(cka_stack.shape[0]):
        col = 0
        for a, b in combinations(tasks, 2):
            pair_mat[li, col] = cka_stack[li, tasks_idx[a], tasks_idx[b]]
            col += 1
    heatmap(pair_mat, pair_names, [f"L{l}" for l in range(cka_stack.shape[0])], "Layerwise CKA pair matrix", os.path.join(PLOTS_DIR, "cka_pairwise_layer_heatmap.png"), xlabel="Task pair", ylabel="Layer", cmap="viridis", annotate=False)

    # Subspace ribbon.
    plot_subspace_ribbon(tasks, overlap_stack, os.path.join(PLOTS_DIR, "subspace_overlap_ribbon.png"))

    # Drift dashboard.
    plot_drift_dashboard(drift_df, os.path.join(PLOTS_DIR, "principal_direction_drift_dashboard.png"))

    # Pairwise centroid distance trajectories.
    pair_rows = []
    for _, r in pair_df.iterrows():
        pair_rows.append({
            "layer": int(r["layer"]),
            "pair": f"{r['task_a']}↔{r['task_b']}",
            "cosine": float(r["cosine"]),
            "l2": float(r["l2"]),
        })
    pair_plot_df = pd.DataFrame(pair_rows)
    save_df(pair_plot_df, os.path.join(CSV_DIR, "pairwise_layer_distance_long.csv"))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), constrained_layout=True)
    for pair, g in pair_plot_df.groupby("pair"):
        axes[0].plot(g["layer"], g["cosine"], marker="o", linewidth=2, label=pair)
        axes[1].plot(g["layer"], g["l2"], marker="o", linewidth=2, label=pair)
    axes[0].set_title("Pairwise centroid cosine across layers")
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("Cosine")
    axes[1].set_title("Pairwise centroid L2 distance across layers")
    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("L2 distance")
    axes[0].legend(frameon=True, fontsize=8)
    axes[1].legend(frameon=True, fontsize=8)
    plt.savefig(os.path.join(PLOTS_DIR, "pairwise_centroid_distance_dashboard.png"), dpi=180, bbox_inches="tight")
    plt.close()

    # Subject-wise ridge plot of final-layer pairwise overlap concentration.
    dist_map = []
    labels = []
    for task in sorted(set(task_labels)):
        sub = df[df["task"] == task]
        C = centroids[task][-1]
        vals = []
        for x in sub["pooled_layers"].tolist():
            xf = np.asarray(x, dtype=np.float64)[-1]
            vals.append(float(np.dot(xf, C) / ((np.linalg.norm(xf) * np.linalg.norm(C)) + 1e-12)))
        dist_map.append(vals)
        labels.append(task)
    ridgeline_plot(dist_map, labels, "Final-layer similarity ridge plot", os.path.join(PLOTS_DIR, "final_layer_similarity_ridges.png"), xlim=(-1, 1))

    # Task counts and accuracies.
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), constrained_layout=True)
    axes[0].bar(task_summary["task"], task_summary["accuracy"], color=plt.get_cmap("tab10")(np.linspace(0.1, 0.9, len(task_summary))))
    axes[0].set_title("Task accuracy")
    axes[0].tick_params(axis="x", rotation=25)
    axes[1].bar(subject_summary["subject"], subject_summary["accuracy"], color=plt.get_cmap("Paired")(np.linspace(0.1, 0.9, len(subject_summary))))
    axes[1].set_title("Subject accuracy")
    axes[1].tick_params(axis="x", rotation=25)
    axes[2].bar(task_summary["task"], task_summary["mean_target_margin"], color=plt.get_cmap("tab10")(np.linspace(0.1, 0.9, len(task_summary))))
    axes[2].set_title("Task mean target margin")
    axes[2].tick_params(axis="x", rotation=25)
    plt.savefig(os.path.join(PLOTS_DIR, "accuracy_and_margin_dashboard.png"), dpi=180, bbox_inches="tight")
    plt.close()

    # Representative embeddings with labels.
    rep_rows = []
    for task in sorted(set(task_labels)):
        sub = df[df["task"] == task]
        if len(sub) == 0:
            continue
        rep = sub.sample(min(10, len(sub)), random_state=SEED)
        for _, r in rep.iterrows():
            rep_rows.append({
                "task": task,
                "question_id": r["sample_id"],
                "question": r["question"],
                "correct": bool(r["correct"]),
                "x": float(emb2d[int(r.name), 0]),
                "y": float(emb2d[int(r.name), 1]),
            })
    save_df(pd.DataFrame(rep_rows), os.path.join(CSV_DIR, "representative_embedding_points.csv"))

    # Big atlas dashboard.
    plot_task_geometry_dashboard(
        centroids,
        sorted(set(task_labels)),
        cka_stack,
        overlap_stack,
        emb2d,
        np.array(task_labels),
        os.path.join(PLOTS_DIR, "task_geometry_atlas_dashboard.png"),
    )

    # Extra: layerwise mean similarity heatmaps.
    pair_mean = np.mean(cka_stack[:, np.triu_indices(len(tasks), 1)[0], np.triu_indices(len(tasks), 1)[1]], axis=1)
    overlap_mean = np.mean(overlap_stack[:, np.triu_indices(len(tasks), 1)[0], np.triu_indices(len(tasks), 1)[1]], axis=1)
    line_with_points(range(len(pair_mean)), [pair_mean, overlap_mean], ["CKA mean", "Subspace overlap mean"], "Layerwise geometry summary", os.path.join(PLOTS_DIR, "layerwise_geometry_summary.png"), xlabel="Layer", ylabel="Score")

    # Write final reports.
    summary = {
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "N_GSM8K": N_GSM8K,
            "N_STRATEGYQA": N_STRATEGYQA,
            "N_MMLU_PER_SUBJECT": N_MMLU_PER_SUBJECT,
            "MMLU_SUBJECTS": MMLU_SUBJECTS,
        },
        "meta": meta,
        "task_summary": task_summary.to_dict(orient="records"),
        "subject_summary": subject_summary.to_dict(orient="records"),
        "drift_summary": drift_df.groupby("task", as_index=False).agg(
            drift_cosine_mean=("drift_cosine", "mean"),
            step_distance_mean=("step_distance", "mean"),
            explained_var1_mean=("explained_var1", "mean"),
        ).to_dict(orient="records"),
    }
    save_json(summary, os.path.join(REPORTS_DIR, "summary.json"))

    md = ["# Task geometry atlas report\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Samples: GSM8K={N_GSM8K}, StrategyQA={N_STRATEGYQA}, MMLU per subject={N_MMLU_PER_SUBJECT}")
    md.append(f"- Subjects: {', '.join(MMLU_SUBJECTS)}")
    md.append("\n## Task summary\n")
    md.append("| Task | Accuracy | Mean target margin | Mean target prob | Mean target rank |")
    md.append("|---|---:|---:|---:|---:|")
    for _, r in task_summary.iterrows():
        md.append(f"| {r['task']} | {r['accuracy']:.3f} | {r['mean_target_margin']:.3f} | {r['mean_target_prob']:.3f} | {r['mean_target_rank']:.2f} |")
    md.append("\n## Subject summary\n")
    md.append("| Subject | Accuracy | Mean target margin | Mean target prob |")
    md.append("|---|---:|---:|---:|")
    for _, r in subject_summary.iterrows():
        md.append(f"| {r['subject']} | {r['accuracy']:.3f} | {r['mean_target_margin']:.3f} | {r['mean_target_prob']:.3f} |")
    md.append("\n## Interpretation\n")
    md.append("- If CKA stays high while subspace overlap drops, tasks share broad features but occupy different effective subspaces.")
    md.append("- If centroid trajectories separate by layer, the model is routing tasks into different representational regions.")
    md.append("- If the final-layer manifold clusters by task more strongly than early layers, task specialization is emerging late.")
    md.append("- If principal-direction drift is small, the model is not strongly reshaping geometry even when outputs differ.")
    md.append("- This supports the story that task performance differences may come from representational routing rather than hard gradient conflict.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    # Console summary.
    print("\n===== TASK SUMMARY =====")
    print(task_summary.to_string(index=False))
    print("\n===== SUBJECT SUMMARY =====")
    print(subject_summary.to_string(index=False))
    print("\nSaved outputs to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOTS_DIR}")
    print(f"- Reports: {REPORTS_DIR}")
    print(f"- NPZ: {NPZ_DIR}")

    free_memory(MODEL)

if __name__ == "__main__":
    main()

2026-06-15 20:19:42.263951: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781554782.438191      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781554782.487970      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781554782.914703      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781554782.914744      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781554782.914747      23 computation_placer.cc:177] computation placer alr

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Generating validation split:   0%|          | 0/29 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Loaded 30 GSM8K, 30 StrategyQA, 60 MMLU samples
Sample 1/120 | gsm8k | gsm8k | Carol and Jennifer are sisters from Los Angeles who love collecting signatures from c...
Sample 2/120 | gsm8k | gsm8k | A team of 4 painters worked on a mansion for 3/8ths of a day every day for 3 weeks. H...
Sample 3/120 | gsm8k | gsm8k | It costs $194 per meter to repave a street. Monica's street is 150 meters long. How m...
Sample 4/120 | gsm8k | gsm8k | Richard lives in an apartment building with 15 floors. Each floor contains 8 units, a...
Sample 5/120 | gsm8k | gsm8k | An ice cream truck is traveling through a neighborhood. Children from various homes h...
Sample 6/120 | gsm8k | gsm8k | James runs 12 miles a day for 5 days a week.  If he runs 10 miles an hour how many ho...
Sample 7/120 | gsm8k | gsm8k | Mark was unwell for 3 months, during which he lost 10 pounds per month. If his final ...
Sample 8/120 | gsm8k | gsm8k | Craig and his brother take turns spelling out the longest letter words they know 

/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_pca.py:789: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = self.explained_variance_ / total_var
/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_pca.py:789: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = self.explained_variance_ / total_var
/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_pca.py:789: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = self.explained_variance_ / total_var
/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_pca.py:789: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = self.explained_variance_ / total_var
/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_pca.py:789: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = self.explained_variance_ / total_var
/usr/


===== TASK SUMMARY =====
      task  n_samples  accuracy  mean_prompt_tokens  mean_completion_tokens  mean_target_margin  mean_target_prob  mean_target_rank
     gsm8k         30  0.166667           69.433333               79.866667          -10.045833          0.000038        169.500000
      mmlu         60  0.316667          106.766667               81.000000          -14.484375          0.000001        709.766667
strategyqa         30  0.500000           25.200000               81.000000          -11.035417          0.000035        576.633333

===== SUBJECT SUMMARY =====
                subject  n_samples  accuracy  mean_prompt_tokens  mean_completion_tokens  mean_target_margin  mean_target_prob  mean_target_rank
       abstract_algebra         12  0.500000           91.333333               81.000000          -15.994792      4.676702e-07        804.666667
    college_mathematics         12  0.250000          107.000000               81.000000          -14.328125      1.539627e-06 